# Session 12 · Data Preparation with scikit-learn

**Data Mining 2026/27** · prof. dr. Uroš Godnov · **Wed 23 Dec 2026 · online (Zoom)** · 3 × 45 min + 15 min break

Real projects spend most of their time here: **missing values, outliers, new features, encodings, leakage**. In AI Studio each of these was an operator in a process. Today you build the same steps in scikit-learn — and package them in a **pipeline** that can never accidentally learn from the test data.

| Part | Time | What we do |
|---|---|---|
| Warm-up | 5 min | 🔁 recall S11 · 🔮 why "split first"? |
| **A — Guided walkthrough** | 80 min | the scikit-learn API · missing data · outliers · feature engineering & encoding · filter selection · pipelines |
| ☕ Break | 15 min | |
| **B — Your turn** | 35 min | prepare the S11 hotel bookings for a cancellation model |
| Wrap-up | 10 min | key takeaways, cheat sheet, exit ticket, what's next |
| 🚀 **C — Advanced track (optional)** | at home, or when done early | your own transformer · leakage you cannot see · target encoding · imputation shoot-out · log targets · shipping a model |

**Learning objectives** — after this session you can:
1. diagnose missing data (structural vs genuinely missing) and choose a strategy: fill, impute or add an indicator;
2. detect outliers (IQR, z-score, domain rules) and decide whether to remove, cap, transform or keep them;
3. engineer features and encode categories (ordinal vs one-hot), and know when scaling is needed;
4. select features with filter methods (`SelectKBest`);
5. build a leak-proof `Pipeline` with `ColumnTransformer` and explain why preprocessing is fitted on training data only;
6. recognise target leakage in a feature set.

**On Zoom.** 🔮 predictions are polls, questions go to the chat. **Lost your connection or runtime?** Click into the current cell → **Runtime → Run before**.

| Icon | Meaning |
|---|---|
| 🔮 | **Predict before you run** |
| 🧪 | **Try it** — 2–5 minutes during the walkthrough |
| ✏️ | **TODO** — your task in Part B |
| ✅ | **Self-check** (⏳ = not attempted yet) |
| 🛟 | **Rescue cell** — run only if stuck, to continue with the next task |
| 🤖 | **AI rule** — try first; then Gemini may *explain* or *hint* |
| 🚀 | **Fast lane** — optional side quests (Part A) and Part C for fluent coders; never graded |
| 📈 💻 | **Two lenses** — a one-minute link to what you already know: 📈 economics and statistics, 💻 programming and databases |

> 💾 **File → Save a copy in Drive** before you start.

**Data today** (from the web): Ames Housing (De Cock, 2011; via OpenML) · Hotel Booking Demand (Antonio et al., 2019).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/urosgodnov/dm-2627-python/blob/main/notebooks/s12-data-preparation.ipynb)

### 🧭 Two lanes — pick yours

| Lane | For whom | How to work through this notebook |
|---|---|---|
| 🟢 **Core** | everyone — especially if Python is still new to you | Part A with the lecturer → Part B → wrap-up. This is everything the assignment and the project need. |
| 🚀 **Fast lane** | you already write Python comfortably (functions, loops, a bit of pandas) — e.g. you passed the 🚀 check in S00 | Follow Part A, and while we cover basics you know, work on the 🚀 **side quests**. Do Part B quickly (it is still the core of the project), then go on to **Part C · 🚀 Advanced track** after the wrap-up. |

You can switch lanes at any time. Everything marked 🚀 is optional, never graded, and its solutions are in the solutions notebook. Use the 📑 table of contents (left sidebar) to jump between parts.

**Two lenses.** 📈 boxes link the topic to economics and statistics, 💻 boxes to programming and databases. Read the one for your programme, or both. In Part C, 📈 challenges need more statistics and less code; 💻 challenges need more programming. **Strong in statistics but new to Python?** Stay in the 🟢 core for the code, and after the wrap-up try the 🧠 think questions and the 📈 challenges.

## 0 · Warm-up (5 min)

### 🔁 Recall from last session (3 min)

Answer **from memory** first — no scrolling back, no AI. Then run the cell below and compare.

1. S11's EDA found two oddities in the hotel bookings: one **impossible value** and one **surprising pattern**. What were they?
2. `adr` (the price per night) has a skewness of about 10.5. Which average and which correlation coefficient do you use for it, and why?
3. Weekend arrivals cancel slightly less often than other bookings (36.3 % vs 37.3 %; p = 0.002, Cramér's V = 0.009). Keep `weekend_arrival` as a predictor?

In [ ]:
#@title 👀 Show the answers { display-mode: "form" }
print("1. 180 bookings with zero guests (impossible, so data errors: today's Part B Task 3 removes them) and Non Refund deposits, 99 % of which were canceled (investigate, e.g. ask the hotel, before you model).")
print('2. The median and Spearman: both are robust to the long right tail and to outliers, while the mean and Pearson are pulled by a few extreme values. Today adds a third option: transform the variable (log).')
print("3. Probably not: the difference is real but negligible, and the tiny p-value comes from the 119,390 rows. Rank features by effect size; today's SelectKBest (A5) automates such scores.")

> 🔮 **Predict before you run:** **Poll:** you will fill missing values with the column median. Should you compute that median on **all** the data or only on the **training** data?

## ⚙️ Setup

In [ ]:
import warnings

import matplotlib.pyplot as plt
import missingno as msno
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn import set_config
from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_openml
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
set_config(display="diagram")            # pipelines are drawn as diagrams
pd.set_option("display.max_columns", 40)
sns.set_theme(style="whitegrid")

def quick_profile(df):
    # Your function from S10.
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_missing": df.isna().sum(),
        "pct_missing": (df.isna().mean() * 100).round(1),
        "n_unique": df.nunique(),
    })

In [ ]:
#@title 🧰 Course helpers — run this cell (no need to read it) { display-mode: "form" }
CHECK_RESULTS = {}  # label -> True (correct) / False (not yet) / None (not attempted)

def check(label, test, hint=""):
    """Give instant feedback on an exercise without ever stopping 'Run all'.

    `test` is a function (usually a lambda) that returns True when the answer is right.
    """
    try:
        ok = bool(test())
    except NameError as err:
        CHECK_RESULTS[label] = None
        print(f"⏳ {label}: not attempted yet ({err}).")
        return
    except Exception as err:  # wrong type, wrong shape, missing column, ...
        CHECK_RESULTS[label] = False
        print(f"❌ {label}: checking your answer raised {type(err).__name__}: {err}")
        if hint:
            print(f"   💡 Hint: {hint}")
        return
    CHECK_RESULTS[label] = ok
    if ok:
        print(f"✅ {label}: correct!")
    else:
        print(f"❌ {label}: not yet.")
        if hint:
            print(f"   💡 Hint: {hint}")

def progress(*, return_count=False):
    """Summarise all self-checks run so far; optional 🚀 fast-lane checks are counted separately."""
    core = [v for label, v in CHECK_RESULTS.items() if not label.startswith("🚀")]
    fast = [v for label, v in CHECK_RESULTS.items() if label.startswith("🚀")]
    solved = sum(1 for v in core if v)
    print(f"Solved {solved} of {len(core)} exercises checked so far.")
    if fast:
        print(f"🚀 Fast lane (optional): solved {sum(1 for v in fast if v)} of {len(fast)}.")
    for label, v in CHECK_RESULTS.items():
        print(f"  {'✅' if v else ('⏳' if v is None else '❌')} {label}")
    if return_count:
        return solved

print("Helpers loaded ✔")

---
# Part A · Guided walkthrough — house prices in Ames, Iowa

## A1 · The scikit-learn API in one picture (8 min)

| Object | Methods | What it does | AI Studio |
|---|---|---|---|
| **Transformer** | `fit` → `transform` | learns parameters from data (a median, a mean, a list of categories), then applies them | Replace Missing Values, Normalize, Nominal to Numerical |
| **Estimator / model** | `fit` → `predict` | learns a model, then predicts | Linear Regression, Decision Tree, … |
| **Pipeline** | `fit` → `predict` | chains transformers + a model into one object | the whole process |

**The golden rule:** split first, then *fit* everything only on the training data. Otherwise you get **leakage**: information from the test data (or from the future) sneaks into model building, and the evaluation looks better than the model really is.

The **Ames Housing** data (De Cock, 2011) describe 1,460 house sales in Ames, Iowa, 2006–2010, with 79 explanatory variables. Target: `SalePrice` (USD).

In [ ]:
ames = fetch_openml(data_id=42165, as_frame=True).frame.drop(columns="Id")
print(ames.shape)
ames.head(3)

In [ ]:
train, test = train_test_split(ames, test_size=0.2, random_state=42)
print("train:", train.shape, " test:", test.shape)

In [ ]:
imputer = SimpleImputer(strategy="median")
imputer.fit(train[["LotFrontage"]])                        # learns the median from TRAIN
print("learned median:", imputer.statistics_)
print("first test values after transform:", imputer.transform(test[["LotFrontage"]])[:5].ravel())

The imputer learned one number (the training median, 70 feet) and fills every gap — in training *and* test data — with it.

> ⚠️ **Common mistake:** `imputer.fit_transform(test)`. It re-learns the median from the test rows (66.5 instead of 70 here): train and test are filled differently, and information from the test data leaks into the preparation. **Rule:** `fit` / `fit_transform` on the training data only; on test (and future) data only `transform`.

## A2 · Missing data (22 min)

In [ ]:
missing = quick_profile(train).query("n_missing > 0").sort_values("pct_missing", ascending=False)
missing

In [ ]:
ax = msno.matrix(train[missing.index], figsize=(12, 4), fontsize=9, sparkline=False)
ax.set_title("Missing values in the training data (white = missing, one row = one house)")
plt.show()

**Missing — or just "none"?** The matrix shows blocks: the garage columns are missing in the *same* houses, and so are the basement columns. The data documentation explains why: `NA` in `PoolQC` means *no pool*, in `Alley` *no alley access*, in `Garage…` *no garage*, in `Bsmt…` *no basement*, in `FireplaceQu` *no fireplace*. These are **structural** "missing" values: the information is there, it is just coded as empty. They should become a category `"None"`, not be imputed. `LotFrontage` (street frontage in feet), by contrast, is **genuinely unknown** for almost 19 % of houses.

Never take this on faith — check it with the data:

In [ ]:
no_garage = train["GarageType"].isna()
print("houses without GarageType:", no_garage.sum(),
      "| of these with GarageArea == 0:", (train.loc[no_garage, "GarageArea"] == 0).sum())

In [ ]:
# 🧪 Try it: Prove that missing `BsmtQual` means "no basement": count how many training houses with missing
#    `BsmtQual` have `TotalBsmtSF == 0`. Store the count in `n_no_basement`.
# missing_bsmt = train["BsmtQual"].isna()
# n_no_basement = ...


In [ ]:
#@title ✅ Check — A2 Try it { display-mode: "form" }
check('A2 Try it', lambda: n_no_basement == train["BsmtQual"].isna().sum() > 0, hint='Filter the rows with a missing BsmtQual, then count TotalBsmtSF == 0.')

In [ ]:
# 🚀 Side quest: Missingness patterns without missingno: which columns are missing TOGETHER? Build `na_groups`, a list
#    of column groups whose missing values sit in exactly the same rows of `train` (use the columns in
#    `missing.index`; keep only groups with 2 or more columns). Then print, for each group, how many houses
#    it affects and what it means. Tip: a column's mask as a tuple, tuple(train[col].isna()), can be a dict key.
# masks = {}
# for col in missing.index:
#     masks.setdefault(..., []).append(col)
# na_groups = [...]


In [ ]:
#@title ✅ Check — 🚀 A2 side quest { display-mode: "form" }
check('🚀 A2 side quest', lambda: sorted(len(g) for g in na_groups if len(g) > 1) == [2, 5, 5], hint='Group the columns by tuple(train[col].isna()); the groups with several columns are the answer.')

**Is "missing" itself informative?** Sometimes *whether* a value is missing predicts the target. scikit-learn can add a 0/1 indicator column next to the imputed values (`add_indicator=True`):

In [ ]:
print(train.groupby(train["LotFrontage"].isna())["SalePrice"].median())

imp = SimpleImputer(strategy="median", add_indicator=True)
pd.DataFrame(imp.fit_transform(train[["LotFrontage"]]),
             columns=["LotFrontage_imputed", "LotFrontage_was_missing"]).head()

Houses with an unknown `LotFrontage` sold for a slightly *higher* median price (171,750 vs 163,000 USD). The gap itself carries a little information, and the indicator column lets a model use it.

> 📈 **Economist's view.** Econometrics names the mechanisms: values can be missing completely at random (MCAR), at random given other observed variables (MAR), or not at random (MNAR, e.g. high earners skipping the income question). For prediction, simple imputation is safest under MCAR; under MAR or MNAR, *whether* a value is missing is itself a signal, which is the case for the indicator column.

> 💻 **Informatics view.** In a database this is NULL semantics: NULL can mean *unknown*, *not applicable* or *none*, and SQL treats it specially (`NULL = NULL` is not true, and `COUNT(col)` skips it). A clean schema keeps those meanings apart, e.g. a `has_garage` flag instead of a NULL `garage_type`, which is what the `"None"` category and the 0/1 flags do.

**Smarter imputation — `KNNImputer`:** fill a missing value with the average of the *k* most similar houses (similarity measured on other columns). Usually more realistic than one global median, but slower, and it needs scaled inputs to measure similarity fairly.

In [ ]:
knn_cols = ["LotFrontage", "LotArea", "GrLivArea", "TotalBsmtSF"]
scaler = StandardScaler().fit(train[knn_cols])
knn = KNNImputer(n_neighbors=5).fit(scaler.transform(train[knn_cols]))
filled = scaler.inverse_transform(knn.transform(scaler.transform(train[knn_cols])))
print("median imputation gives every house:", train["LotFrontage"].median())
print("KNN imputation, first five filled values:", filled[train["LotFrontage"].isna().to_numpy(), 0][:5].round(1))

Each missing frontage now gets its own estimate from five similar houses, instead of the same 70 feet for everyone. Does that make the *model* better? Measure it — the 🚀 C4 challenge does.

## A3 · Outliers (15 min)

In [ ]:
big = train["GrLivArea"] > 4000
plt.figure(figsize=(8, 5))
plt.scatter(train["GrLivArea"], train["SalePrice"], s=8, alpha=0.5)
plt.scatter(train.loc[big, "GrLivArea"], train.loc[big, "SalePrice"], s=60, color="red")
plt.xlabel("above-ground living area (sq ft)")
plt.ylabel("SalePrice (USD)")
plt.title("Three very large houses (red) — two of them very cheap")
plt.show()
train.loc[big, ["GrLivArea", "SalePrice", "SaleCondition"]]

> 🔮 **Predict before you run:** The classic **IQR rule** flags values beyond Q1 − 1.5·IQR or Q3 + 1.5·IQR. How many of the 1,168 training houses will it flag on `GrLivArea` — just the red ones?

In [ ]:
def iqr_bounds(s, k=1.5):
    q1, q3 = s.quantile([0.25, 0.75])
    return q1 - k * (q3 - q1), q3 + k * (q3 - q1)

lo, hi = iqr_bounds(train["GrLivArea"])
print(f"IQR rule: {((train['GrLivArea'] < lo) | (train['GrLivArea'] > hi)).sum()} houses outside {lo:.0f}–{hi:.0f} sq ft")
z = (train["GrLivArea"] - train["GrLivArea"].mean()) / train["GrLivArea"].std()
print("z-score rule (|z| > 3):", (z.abs() > 3).sum(), "houses")

In [ ]:
# 🧪 Try it: `GrLivArea` is right-skewed. How many training houses does the IQR rule flag on the LOG of
#    `GrLivArea`? Reuse `iqr_bounds` and store the number in `n_flag_log`.
# log_area = np.log(train["GrLivArea"])
# lo_log, hi_log = iqr_bounds(log_area)
# n_flag_log = ...


In [ ]:
#@title ✅ Check — A3 Try it { display-mode: "form" }
check('A3 Try it', lambda: n_flag_log == 7, hint="Same rule on np.log(train['GrLivArea']): count values below the lower OR above the upper bound.")

On the log scale far fewer houses look extreme: many "outliers" of a skewed variable are just its long right tail.

**Domain knowledge decides.** The two cheap giants are **partial sales** (`SaleCondition = Partial`): houses sold before completion, at prices that do not reflect the market. De Cock (2011) himself recommends excluding such cases. Options in general: **remove**, **cap** (*winsorise*: replace extreme values by a limit, e.g. `s.clip(upper=hi)`), **transform** (e.g. log), or **keep** them and use a robust model.

In [ ]:
train_clean = train[~((train["GrLivArea"] > 4000) & (train["SalePrice"] < 300_000))].copy()
print(len(train), "→", len(train_clean), "training houses")

> 🔮 **Predict before you run:** Should we delete such houses from the **test** set, too?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
sns.histplot(train_clean["SalePrice"], ax=axes[0])
axes[0].set(title=f"SalePrice — skewness {train_clean['SalePrice'].skew():.2f}",
            xlabel="SalePrice (USD)", ylabel="number of houses")
sns.histplot(np.log1p(train_clean["SalePrice"]), ax=axes[1])
axes[1].set(title=f"log(1 + SalePrice) — skewness {np.log1p(train_clean['SalePrice']).skew():.2f}",
            xlabel="log(1 + SalePrice)", ylabel="number of houses")
plt.tight_layout()
plt.show()

Prices are right-skewed (skewness 1.74); on the log scale they are almost symmetric (0.13), and errors become *relative* (10 % off is equally bad for a cheap and an expensive house). We model `log(1 + SalePrice)` from now on.

> 📈 **Economist's view.** With log price on log living area, a regression coefficient *b* is an elasticity: 1 % more space goes with about *b* % higher price, the language of hedonic price models. Capping extremes (winsorising, e.g. at the 1st and 99th percentiles) is standard practice in finance and empirical economics, where a few extreme returns or incomes would otherwise dominate the estimates.

> 💻 **Informatics view.** In an ETL job an outlier rule is a data-quality rule: a check such as Part B's `0 <= adr <= 1000` rejects a record, sends it to a quarantine table for a human, or flags it, and it logs how often it fired. Keep such rules in code, under version control and next to the pipeline, like a schema constraint.

In [ ]:
# 🚀 Side quest: Multivariate outliers: fit `IsolationForest(contamination=0.01, random_state=42)` on
#    train[["GrLivArea", "SalePrice"]] and store its flags in `iso_flag`, a boolean Series (True = outlier)
#    with the same index as `train` (predict() returns -1 for outliers). How many houses does it flag, how many
#    of them does the IQR rule on GrLivArea flag too, and which house does only IsolationForest catch?
# from sklearn.ensemble import IsolationForest
# iso = IsolationForest(contamination=0.01, random_state=42).fit(...)
# iso_flag = pd.Series(..., index=train.index)


In [ ]:
#@title ✅ Check — 🚀 A3 side quest { display-mode: "form" }
check('🚀 A3 side quest', lambda: iso_flag.dtype == bool and len(iso_flag) == len(train) and 5 <= iso_flag.sum() <= 30 and iso_flag.loc[(train["GrLivArea"] > 4000) & (train["SalePrice"] < 300_000)].all(), hint='pd.Series(iso.predict(X) == -1, index=train.index); both cheap giants should be flagged.')

## A4 · Feature engineering and encoding (20 min)

**Stateless** transformations (formulas within one row: sums, ratios, ages) learn nothing from other rows, so you can apply them to train and test alike. **Stateful** ones (imputers, scalers, encoders, selectors) learn from data and must be fitted on the training data only — that is what pipelines are for.

In [ ]:
def add_features(df):
    df = df.copy()
    df["TotalSF"] = df["TotalBsmtSF"] + df["1stFlrSF"] + df["2ndFlrSF"]
    df["HouseAge"] = df["YrSold"] - df["YearBuilt"]
    df["YearsSinceRemodel"] = df["YrSold"] - df["YearRemodAdd"]
    df["TotalBath"] = df["FullBath"] + 0.5 * df["HalfBath"] + df["BsmtFullBath"] + 0.5 * df["BsmtHalfBath"]
    df["HasGarage"] = (df["GarageArea"] > 0).astype(int)
    return df

train_fe = add_features(train_clean)
test_fe = add_features(test)
y_log = np.log1p(train_fe["SalePrice"])

for col in ["GrLivArea", "TotalBsmtSF", "TotalSF", "FullBath", "TotalBath", "YearBuilt", "HouseAge"]:
    print(f"{col:18s} correlation with log price: {train_fe[col].corr(y_log):+.3f}")

`TotalSF` correlates more strongly than any of its parts — a good engineered feature combines information the model would otherwise have to discover by itself.

**Encoding categories.** Models need numbers.
- **Ordinal** categories have an order — quality ratings `Po < Fa < TA < Gd < Ex` → `OrdinalEncoder` with the order written out.
- **Nominal** categories have none — `Neighborhood` → `OneHotEncoder` (one 0/1 column per category; `handle_unknown="ignore"` for categories first seen in new data).

In [ ]:
QUALITY = ["None", "Po", "Fa", "TA", "Gd", "Ex"]
enc = OrdinalEncoder(categories=[QUALITY])
sample = train_fe[["KitchenQual"]].head(6)
sample.assign(encoded=enc.fit_transform(sample).ravel())

> ⚠️ **Common mistake:** `OrdinalEncoder()` without `categories=`. It sorts the labels alphabetically (`Ex < Fa < Gd < Po < TA`), so "Excellent" gets the *lowest* code. Always write the order out.

In [ ]:
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(train_fe[["Neighborhood"]])
names = ohe.get_feature_names_out()
print(len(names), "0/1 columns, e.g.", list(names[:4]))

**Scaling** (`StandardScaler`: mean 0, standard deviation 1) matters for models that compare magnitudes or distances — linear models with regularisation, k-NN, SVM, neural networks. Tree-based models (decision trees, random forests, XGBoost) do not need it.

In [ ]:
# 🧪 Try it: Create a copy of `train_fe` called `train_try` with a new feature `TotalPorchSF` = OpenPorchSF +
#    EnclosedPorch + 3SsnPorch + ScreenPorch + WoodDeckSF. Store its correlation with `y_log` in `porch_corr`.
# train_try = train_fe.copy()
# train_try["TotalPorchSF"] = ...
# porch_corr = ...


In [ ]:
#@title ✅ Check — A4 Try it { display-mode: "form" }
check('A4 Try it', lambda: "TotalPorchSF" in train_try.columns and abs(porch_corr - 0.397) < 0.01, hint='Add the five columns, then .corr(y_log).')

In [ ]:
# 🚀 Side quest: Feature engineering can live inside a pipeline, too. (1) Wrap `add_features` in a `FunctionTransformer`
#    called `fe_step`, fit it on `train_clean` and confirm that it turns `test` into exactly `test_fe` (it learns
#    nothing: stateless). (2) Store in `skewed_cols` the numeric columns of `train_fe` (without SalePrice) whose
#    skewness is above 1, and log-transform them with FunctionTransformer(np.log1p, feature_names_out="one-to-one")
#    into a DataFrame `train_logged`. Which features correlate more strongly with `y_log` after the log?
# from sklearn.preprocessing import FunctionTransformer
# fe_step = FunctionTransformer(add_features).fit(train_clean)
# skewed_cols = ...
# train_logged = ...


In [ ]:
#@title ✅ Check — 🚀 A4 side quest { display-mode: "form" }
check('🚀 A4 side quest', lambda: fe_step.transform(test).equals(test_fe) and "LotArea" in skewed_cols and train_logged["LotArea"].corr(y_log) > 0.4, hint='num.skew() > 1 gives a boolean mask over the columns; np.log1p keeps the DataFrame and its names.')

## A5 · Filter feature selection (7 min)

S11 used statistics to judge one feature at a time. `SelectKBest` does exactly that automatically: it scores every feature against the target and keeps the *k* best. Two common scores:
- **F score** (`f_regression`): how well a straight line through the feature explains the target — large = strong *linear* relationship;
- **mutual information**: how much knowing the feature reduces the uncertainty about the target — 0 = independent; it also catches *non-linear* relationships.

In [ ]:
num_features = train_fe.select_dtypes("number").drop(columns="SalePrice").columns
X_num = SimpleImputer(strategy="median").fit_transform(train_fe[num_features])

selector = SelectKBest(score_func=f_regression, k=10).fit(X_num, y_log)
f_scores = pd.Series(selector.scores_, index=num_features).sort_values(ascending=False)
mi_scores = pd.Series(mutual_info_regression(X_num, y_log, random_state=42), index=num_features)
(pd.DataFrame({"F score": f_scores.round(0), "mutual information": mi_scores.round(3)})
 .sort_values("F score", ascending=False)
 .head(10))

Both scores agree on the top three — and our engineered `TotalSF` ranks first, ahead of `OverallQual` and `GrLivArea`.

**Caveats.** Filter methods judge each feature *alone*: they miss interactions and happily keep redundant pairs (`GrLivArea` **and** `TotalSF`). They also learn from the target, so inside a real workflow they belong **in the pipeline**, fitted on training folds only. Model-based selection (Lasso, tree importances) follows in S13.

> ⚠️ **Common mistake:** running `SelectKBest` (or a scaler or an imputer) on the full dataset and *then* cross-validating the model. The selection has already seen the validation folds, so the score is too optimistic. Put the selector inside the pipeline. (🚀 C2 shows how extreme it gets: on pure noise, selection outside cross-validation reports an AUC of about 0.95.)

## A6 · Pipelines: every step in one leak-proof object (8 min)

A `ColumnTransformer` sends each group of columns through its own preparation; a `Pipeline` chains that preparation with a model. When you cross-validate the pipeline, **every step is re-fitted on each training fold** — leakage becomes impossible by construction. In AI Studio terms: the preparation operators sit **inside** the *Cross Validation* operator, not in front of it.

We measure the error with **RMSE** (root mean squared error, the typical size of a prediction error). On the log scale, an RMSE of 0.10 means a typical error of about 10 % of the price. **5-fold cross-validation** trains on four fifths of the training data and validates on the remaining fifth, five times; we report the mean and the spread (±).

> 📈 **Economist's view.** A pipeline is like a Stata do-file that rebuilds every variable from the raw data, with one extra guarantee: cross-validation re-runs it on each training fold, so no step can peek at the rows it is later judged on. That makes the whole preparation reproducible and auditable, like a replication package.

> 💻 **Informatics view.** A fitted pipeline is a deployable function (raw rows in, predictions out, every learned parameter inside), like an ETL job with a fixed, versioned transformation. Save it with `joblib` and load it in a batch job or an API, and every step inside it runs exactly as in training (🚀 C6 does this).

In [ ]:
ORDINAL_COLS = ["ExterQual", "ExterCond", "BsmtQual", "BsmtCond", "HeatingQC",
                "KitchenQual", "FireplaceQu", "GarageQual", "GarageCond", "PoolQC"]
NOMINAL_COLS = [c for c in train_fe.select_dtypes(exclude="number").columns if c not in ORDINAL_COLS]
NUMERIC_COLS = [c for c in train_fe.select_dtypes("number").columns if c != "SalePrice"]

numeric = Pipeline([("impute", SimpleImputer(strategy="median", add_indicator=True)),
                    ("scale", StandardScaler())])
ordinal = Pipeline([("fill_none", SimpleImputer(strategy="constant", fill_value="None")),
                    ("encode", OrdinalEncoder(categories=[QUALITY] * len(ORDINAL_COLS),
                                              handle_unknown="use_encoded_value", unknown_value=-1))])
nominal = Pipeline([("fill_none", SimpleImputer(strategy="constant", fill_value="None")),
                    ("encode", OneHotEncoder(handle_unknown="ignore"))])

preprocess = ColumnTransformer([("num", numeric, NUMERIC_COLS),
                                ("ord", ordinal, ORDINAL_COLS),
                                ("nom", nominal, NOMINAL_COLS)])
model = Pipeline([("prep", preprocess), ("reg", Ridge(alpha=10))])   # Ridge = linear regression with a brake (S13)
model

> 🔮 **Predict before you run:** A baseline uses only three raw features (`GrLivArea`, `OverallQual`, `YearBuilt`). By how much will the full pipeline reduce the cross-validated error?

In [ ]:
X_train = train_fe.drop(columns="SalePrice")

baseline = Pipeline([("prep", ColumnTransformer([("num", SimpleImputer(strategy="median"),
                                                  ["GrLivArea", "OverallQual", "YearBuilt"])])),
                     ("reg", LinearRegression())])
for name, pipe in [("baseline (3 features)", baseline), ("full pipeline", model)]:
    scores = cross_val_score(pipe, X_train, y_log, cv=5, scoring="neg_root_mean_squared_error")
    print(f"{name:22s} CV RMSE (log scale) = {-scores.mean():.3f} ± {scores.std():.3f}")

The full pipeline cuts the error by about a third. The ± spread shows how much the score moves from fold to fold: differences smaller than that are hard to trust.

In [ ]:
model.fit(X_train, y_log)
pred = model.predict(test_fe.drop(columns="SalePrice"))
rmse_test = np.sqrt(np.mean((pred - np.log1p(test_fe["SalePrice"])) ** 2))
print(f"test RMSE (log scale): {rmse_test:.3f}  — evaluated once, at the very end")

The test error (0.126) is a little higher than the cross-validated 0.114 ± 0.008 — normal for a single test set of 292 houses, and no sign of overfitting: the pipeline generalises. Resist the urge to tune against it: the test set is used **once**. In S13 we open the model box itself — linear regression, regularisation, trees — and learn to tell **underfitting** from **overfitting**.

In [ ]:
# 🚀 Side quest: Look inside the fitted pipeline. (1) Get the preprocessed training data as a DataFrame `prep_df` with
#    readable column names: clone `preprocess`, switch its one-hot encoder to dense output with
#    set_params(nom__encode__sparse_output=False), then .set_output(transform="pandas") and .fit_transform(X_train).
#    (Why is the switch needed? Try without it.) (2) Store in `top_coefs` the 5 Ridge coefficients of the fitted
#    `model` with the largest absolute values, indexed by feature name (model.named_steps["prep"].get_feature_names_out()).
# from sklearn.base import clone
# prep_df = clone(preprocess).set_params(...).set_output(transform="pandas").fit_transform(X_train)
# coefs = pd.Series(model.named_steps["reg"].coef_, index=...)
# top_coefs = ...


In [ ]:
#@title ✅ Check — 🚀 A6 side quest { display-mode: "form" }
check('🚀 A6 side quest', lambda: prep_df.shape == (1166, 268) and len(top_coefs) == 5 and top_coefs.index[0] == "nom__Neighborhood_Crawfor", hint='Sort coefs.abs() in descending order and take the first five index labels; coefs[...] keeps the signs.')

> **Before the break:** read the Part B scenario below.

---
## ☕ Break — 15 minutes

---
# Part B · Your turn: prepare the hotel bookings for a cancellation model (35 min)

**Scenario.** The revenue manager of the two hotels from S11 wants to predict, **at booking time**, which bookings will be canceled. Your job: turn the raw export into a clean, leak-free feature table and a first cross-validated baseline. Each task builds a new table — `hotels_nl` → `hotels_mv` → `hotels_q` → `hotels_fe`.

💡 Tasks 4–5 can be finished at home; S13 starts from a pipeline like yours.

In [ ]:
HOTELS_URL = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-02-11/hotels.csv"
hotels = pd.read_csv(HOTELS_URL)
pd.crosstab(hotels["reservation_status"], hotels["is_canceled"])

In [ ]:
# ✏️ TODO: Task 1 — Leakage. The crosstab above shows that `reservation_status` reveals the answer. Create
#    `hotels_nl` without the two columns that are only known AFTER the stay or cancellation:
#    `reservation_status` and `reservation_status_date`.
# hotels_nl = hotels.drop(columns=[...])


In [ ]:
#@title ✅ Check — Task 1 { display-mode: "form" }
check('Task 1', lambda: hotels_nl.shape == (119390, 30) and "reservation_status" not in hotels_nl.columns, hint='Drop both columns with .drop(columns=[...]).')

**✍️ Your answer** — Why is `reservation_status` leakage, and can you spot a 'grey-zone' column that may also change after the booking was made?

> *(double-click this cell and write your answer here)*

In [ ]:
#@title 🛟 Rescue — run only if you could not finish Task 1 { display-mode: "form" }
hotels_nl = hotels.drop(columns=["reservation_status", "reservation_status_date"])
print("hotels_nl ready:", hotels_nl.shape)

In [ ]:
# ✏️ TODO: Task 2 — Missing values. Create `hotels_mv` from `hotels_nl`: fill `children` with 0 and `country`
#    with "Unknown"; add 0/1 flags `has_agent` and `has_company` (1 if the value is present), then drop
#    `agent` and `company` (structural missingness: most guests book without them).
# hotels_mv = hotels_nl.copy()
# hotels_mv["children"] = ...
# ...


In [ ]:
#@title ✅ Check — Task 2 { display-mode: "form" }
check('Task 2', lambda: hotels_mv.isna().sum().sum() == 0 and {"has_agent", "has_company"} <= set(hotels_mv.columns) and "company" not in hotels_mv.columns, hint='.notna().astype(int) creates a 0/1 flag; then drop the original columns.')

In [ ]:
#@title 🛟 Rescue — run only if you could not finish Task 2 { display-mode: "form" }
hotels_mv = hotels_nl.copy()
hotels_mv["children"] = hotels_mv["children"].fillna(0)
hotels_mv["country"] = hotels_mv["country"].fillna("Unknown")
hotels_mv["has_agent"] = hotels_mv["agent"].notna().astype(int)        # structural: booked without an agent
hotels_mv["has_company"] = hotels_mv["company"].notna().astype(int)    # structural: booked without a company
hotels_mv = hotels_mv.drop(columns=["agent", "company"])
print("hotels_mv ready:", hotels_mv.shape)

In [ ]:
# ✏️ TODO: Task 3 — Impossible and extreme values. Create `hotels_q` from `hotels_mv`, keeping only bookings
#    with at least one guest (adults + children + babies > 0) and an `adr` between 0 and 1000.
# guests = ...
# hotels_q = hotels_mv[(...) & (...)].copy()


In [ ]:
#@title ✅ Check — Task 3 { display-mode: "form" }
check('Task 3', lambda: len(hotels_q) == 119208 and hotels_q["adr"].between(0, 1000).all(), hint='.between(0, 1000) keeps both ends; combine the two conditions with &.')

In [ ]:
#@title 🛟 Rescue — run only if you could not finish Task 3 { display-mode: "form" }
guests = hotels_mv["adults"] + hotels_mv["children"] + hotels_mv["babies"]
hotels_q = hotels_mv[(guests > 0) & hotels_mv["adr"].between(0, 1000)].copy()
print("hotels_q ready:", hotels_q.shape)

In [ ]:
# ✏️ TODO: Task 4 — Feature engineering. Create `hotels_fe` from `hotels_q` with four new columns:
#    `total_nights`, `total_guests`, `is_family` (1 if children + babies > 0) and `arrival_month` (1–12,
#    from the month name — tip: pd.to_datetime(..., format="%B").dt.month).
# hotels_fe = hotels_q.copy()
# hotels_fe["total_nights"] = ...


In [ ]:
#@title ✅ Check — Task 4 { display-mode: "form" }
check('Task 4', lambda: {"total_nights", "total_guests", "is_family", "arrival_month"} <= set(hotels_fe.columns) and hotels_fe["arrival_month"].between(1, 12).all() and hotels_fe["total_guests"].min() > 0, hint='pd.to_datetime(hotels_fe["arrival_date_month"], format="%B").dt.month turns "July" into 7.')

In [ ]:
#@title 🛟 Rescue — run only if you could not finish Task 4 { display-mode: "form" }
hotels_fe = hotels_q.copy()
hotels_fe["total_nights"] = hotels_fe["stays_in_weekend_nights"] + hotels_fe["stays_in_week_nights"]
hotels_fe["total_guests"] = hotels_fe["adults"] + hotels_fe["children"] + hotels_fe["babies"]
hotels_fe["is_family"] = ((hotels_fe["children"] + hotels_fe["babies"]) > 0).astype(int)
hotels_fe["arrival_month"] = pd.to_datetime(hotels_fe["arrival_date_month"], format="%B").dt.month
print("hotels_fe ready:", hotels_fe.shape)

In [ ]:
NUM_H = ["lead_time", "total_nights", "total_guests", "adr", "previous_cancellations",
         "previous_bookings_not_canceled", "booking_changes", "days_in_waiting_list",
         "required_car_parking_spaces", "total_of_special_requests", "arrival_month",
         "is_repeated_guest", "is_family", "has_agent", "has_company"]
CAT_H = ["hotel", "meal", "country", "market_segment", "distribution_channel",
         "deposit_type", "customer_type", "reserved_room_type"]
sample = hotels_fe.sample(30_000, random_state=42)     # a sample keeps cross-validation fast
print(sample.shape)

In [ ]:
# ✏️ TODO: Task 5 — A leak-proof baseline. Build a Pipeline `hotel_pipe`: a ColumnTransformer that scales
#    NUM_H (StandardScaler) and one-hot encodes CAT_H (OneHotEncoder(handle_unknown="ignore",
#    min_frequency=50) — rare countries are grouped), followed by LogisticRegression(max_iter=1000).
#    Store the mean 5-fold cross-validated ROC-AUC on `sample` in `auc_baseline`.
# hotel_pipe = Pipeline([
#     ("prep", ColumnTransformer([...])),
#     ("clf", LogisticRegression(max_iter=1000)),
# ])
# auc_baseline = cross_val_score(..., scoring="roc_auc").mean()


In [ ]:
#@title ✅ Check — Task 5 { display-mode: "form" }
check('Task 5', lambda: 0.85 < auc_baseline < 0.93, hint='cross_val_score(hotel_pipe, sample[NUM_H + CAT_H], sample["is_canceled"], cv=5, scoring="roc_auc").mean()')

**✍️ Your answer** — Your baseline reaches a ROC-AUC of about 0.89. What does that mean, and is it good?

> *(double-click this cell and write your answer here)*

In [ ]:
# ✏️ ⭐ Stretch: See leakage with your own eyes: repeat Task 5 with `reservation_status` added as an extra
#    categorical feature (take it from the original `hotels` table). Store the AUC in `auc_leak`.
# sample_leak = sample.join(hotels["reservation_status"])
# ...


In [ ]:
#@title ✅ Check — Stretch { display-mode: "form" }
check('Stretch', lambda: auc_leak > 0.99, hint="sample.join(hotels['reservation_status']) aligns rows by index.")

> 🚀 **Done early?** Part C · Advanced track waits right after the wrap-up (📑 table of contents → *Part C*). Run its ⚙️ setup cell first, then pick any challenge.

---
## Wrap-up (10 min)

**Key takeaways**
1. **Split first.** Anything learned from data — medians, scaling parameters, categories, selected features — is learned on training data only.
2. Missing ≠ missing: structural "missing" values are information (`"None"`, 0/1 flags); genuinely unknown values are imputed — optionally with an indicator.
3. Outlier rules produce candidates; domain knowledge makes the decision — and you never delete test rows.
4. Good features (`TotalSF`) and correct encodings (ordinal vs one-hot) often beat fancier models.
5. A `Pipeline` + `ColumnTransformer` makes preparation reproducible and leak-proof; a "perfect" score (AUC = 1.0) is a red flag, not a triumph.

In [ ]:
progress()

### 📋 Cheat sheet — today's essentials

| I want to … | Code |
|---|---|
| learn on train, apply to test | `imp.fit(X_train); imp.transform(X_test)` |
| fill gaps and flag them | `SimpleImputer(strategy="median", add_indicator=True)` |
| turn structural NA into a category | `SimpleImputer(strategy="constant", fill_value="None")` |
| impute from similar rows (scale first) | `KNNImputer(n_neighbors=5)` |
| outlier candidates (IQR rule) | `q1, q3 = s.quantile([0.25, 0.75]); limits q1 - 1.5*(q3 - q1), q3 + 1.5*(q3 - q1)` |
| tame a right-skewed target | `y = np.log1p(price); back to dollars: np.expm1(pred)` |
| encode ordered categories | `OrdinalEncoder(categories=[["None", "Po", "Fa", "TA", "Gd", "Ex"]])` |
| encode unordered categories | `OneHotEncoder(handle_unknown="ignore", min_frequency=50)` |
| keep the k best features | `SelectKBest(f_regression, k=10)` |
| different preparation per column group | `ColumnTransformer([("num", num_pipe, NUM), ("cat", cat_pipe, CAT)])` |
| preparation + model in one leak-proof object | `Pipeline([("prep", preprocess), ("model", Ridge())])` |
| honest score of the whole pipeline | `cross_val_score(pipe, X, y, cv=5, scoring="roc_auc")` |

> 🧩 **For your project:** before you train anything, (1) list the columns that are known **at prediction time** and drop the rest — that is your leakage check; (2) decide per column: structural NA → category or 0/1 flag, genuinely missing → impute *inside* the pipeline; (3) write down every outlier decision with its reason (domain knowledge, not just a rule) and never delete test rows; (4) put every learned step in one `Pipeline` and report its cross-validated score as your baseline.

**✍️ Your answer** — **Exit ticket.** (1) Name one stateless and one stateful preparation step. (2) Why must a scaler live inside the pipeline? (3) What should you do when a model scores a perfect AUC of 1.0?

> *(double-click this cell and write your answer here)*

**Next time — S13 (Mon 4 / Wed 6 Jan, in class):** linear and logistic regression, decision trees, and the central question of machine learning — **underfitting or overfitting?** Merry Christmas and a happy New Year! 🎄

## 🧠 Think it through (optional · no code · 10 min)

Questions for thinking rather than coding, e.g. if you are strong in statistics but new to Python. Discuss them in pairs, or write two or three sentences for each. The solutions notebook has model answers.

1. 📈 The hotel wants to demand a deposit from bookings that the Part B model flags as likely to cancel. Why can the model not tell you whether a deposit *causes* fewer cancellations, and what would you need to find out?
2. 📈 You fill the 18.6 % missing `LotFrontage` values with the median (70 ft) and then run an OLS regression to estimate how frontage affects price. What goes wrong with the coefficient and its standard error?
3. 💻 Your pipeline was trained on the cleaned Part B table, but a nightly job scores raw bookings straight from the booking database. Name two things that can silently go wrong, and how pipeline design protects you.
4. 💻 `reservation_status` was easy to spot. Where else can leakage hide when you build features with SQL from operational tables, and how do you prevent it systematically?

> ✍️ *(double-click this cell and write your answers here)*

---
# Part C · 🚀 Advanced track (optional)

**For:** fast-lane students who finished Part B — or anyone who wants more, at home. Never graded; the assignment and the project do not need it.

**Theme:** Build your own scikit-learn components — and catch the leakage that hides inside good-looking scores.

| # | Challenge | Time | Level | Lens |
|---|---|---|---|---|
| **C1** | Write your own transformer: `RareCategoryGrouper` | 15 min | 🌶️🌶️ | 💻 |
| **C2** | Selection outside cross-validation invents signal | 10 min | 🌶️🌶️ | 📈💻 |
| **C3** | Target encoding, the right way | 15 min | 🌶️🌶️ | 💻 |
| **C4** | Imputation shoot-out | 15 min | 🌶️🌶️ | 📈 |
| **C5** | Train in logs, report in dollars | 10 min | 🌶️ | 📈 |
| **C6** | Ship the pipeline: save, load and validate | 15 min | 🌶️🌶️ | 💻 |
| **🏁** | Open: beat the Part A pipeline without leakage | 20 min | open | 📈💻 |

**Lens:** 📈 more statistics and economics, less code · 💻 more programming and data engineering · 📈💻 both

Each challenge stands on its own. **Run the ⚙️ Part C setup cell first**, then pick any challenge. The usual rules apply: a first own attempt, then Gemini may *explain* or give a *hint*.

The setup cell rebuilds the Ames split and pipeline pieces, the prepared hotel sample (`hotels_c`, the same 30,000 bookings as `sample` in Part B) and the data for C2, C3 and C6. After a runtime restart it downloads both datasets again, so it takes a little longer.

In [ ]:
#@title ⚙️ Part C setup — run this first (rebuilds everything Part C needs) { display-mode: "form" }
if "check" not in globals():  # runtime restarted? then restore the ✅ self-check helpers
    CHECK_RESULTS = {}  # label -> True (correct) / False (not yet) / None (not attempted)

    def check(label, test, hint=""):
        """Give instant feedback on an exercise without ever stopping 'Run all'.

        `test` is a function (usually a lambda) that returns True when the answer is right.
        """
        try:
            ok = bool(test())
        except NameError as err:
            CHECK_RESULTS[label] = None
            print(f"⏳ {label}: not attempted yet ({err}).")
            return
        except Exception as err:  # wrong type, wrong shape, missing column, ...
            CHECK_RESULTS[label] = False
            print(f"❌ {label}: checking your answer raised {type(err).__name__}: {err}")
            if hint:
                print(f"   💡 Hint: {hint}")
            return
        CHECK_RESULTS[label] = ok
        if ok:
            print(f"✅ {label}: correct!")
        else:
            print(f"❌ {label}: not yet.")
            if hint:
                print(f"   💡 Hint: {hint}")

    def progress(*, return_count=False):
        """Summarise all self-checks run so far; optional 🚀 fast-lane checks are counted separately."""
        core = [v for label, v in CHECK_RESULTS.items() if not label.startswith("🚀")]
        fast = [v for label, v in CHECK_RESULTS.items() if label.startswith("🚀")]
        solved = sum(1 for v in core if v)
        print(f"Solved {solved} of {len(core)} exercises checked so far.")
        if fast:
            print(f"🚀 Fast lane (optional): solved {sum(1 for v in fast if v)} of {len(fast)}.")
        for label, v in CHECK_RESULTS.items():
            print(f"  {'✅' if v else ('⏳' if v is None else '❌')} {label}")
        if return_count:
            return solved

    print("Helpers loaded ✔")

import warnings

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.datasets import fetch_openml
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.experimental import enable_iterative_imputer  # noqa: F401 (unlocks IterativeImputer)
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import IterativeImputer, KNNImputer, SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge, RidgeCV
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (FunctionTransformer, OneHotEncoder, OrdinalEncoder, StandardScaler,
                                   TargetEncoder)
from sklearn.utils.validation import check_is_fitted

warnings.filterwarnings("ignore", category=FutureWarning)

# ---- Ames (Part A): split, outlier rule, features, preprocessing
if "ames" not in globals():
    ames = fetch_openml(data_id=42165, as_frame=True).frame.drop(columns="Id")
QUALITY = ["None", "Po", "Fa", "TA", "Gd", "Ex"]
ORDINAL_COLS = ["ExterQual", "ExterCond", "BsmtQual", "BsmtCond", "HeatingQC",
                "KitchenQual", "FireplaceQu", "GarageQual", "GarageCond", "PoolQC"]

def add_features(df):
    df = df.copy()
    df["TotalSF"] = df["TotalBsmtSF"] + df["1stFlrSF"] + df["2ndFlrSF"]
    df["HouseAge"] = df["YrSold"] - df["YearBuilt"]
    df["YearsSinceRemodel"] = df["YrSold"] - df["YearRemodAdd"]
    df["TotalBath"] = df["FullBath"] + 0.5 * df["HalfBath"] + df["BsmtFullBath"] + 0.5 * df["BsmtHalfBath"]
    df["HasGarage"] = (df["GarageArea"] > 0).astype(int)
    return df

train, test = train_test_split(ames, test_size=0.2, random_state=42)
train_clean = train[~((train["GrLivArea"] > 4000) & (train["SalePrice"] < 300_000))].copy()
train_fe, test_fe = add_features(train_clean), add_features(test)
X_train, y_log = train_fe.drop(columns="SalePrice"), np.log1p(train_fe["SalePrice"])
NOMINAL_COLS = [c for c in train_fe.select_dtypes(exclude="number").columns if c not in ORDINAL_COLS]
NUMERIC_COLS = [c for c in train_fe.select_dtypes("number").columns if c != "SalePrice"]

def make_preprocess(numeric=None):
    """A fresh, unfitted copy of the Part A preprocessing; `numeric` replaces its numeric pipeline."""
    if numeric is None:
        numeric = Pipeline([("impute", SimpleImputer(strategy="median", add_indicator=True)),
                            ("scale", StandardScaler())])
    ordinal = Pipeline([("fill_none", SimpleImputer(strategy="constant", fill_value="None")),
                        ("encode", OrdinalEncoder(categories=[QUALITY] * len(ORDINAL_COLS),
                                                  handle_unknown="use_encoded_value", unknown_value=-1))])
    nominal = Pipeline([("fill_none", SimpleImputer(strategy="constant", fill_value="None")),
                        ("encode", OneHotEncoder(handle_unknown="ignore"))])
    return ColumnTransformer([("num", numeric, NUMERIC_COLS), ("ord", ordinal, ORDINAL_COLS),
                              ("nom", nominal, NOMINAL_COLS)])

cv_rmse_partA = -cross_val_score(Pipeline([("prep", make_preprocess()), ("reg", Ridge(alpha=10))]),
                                 X_train, y_log, cv=5, scoring="neg_root_mean_squared_error").mean()

# ---- Hotels (Part B): the prepared 30,000-booking sample
HOTELS_URL = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-02-11/hotels.csv"
if "hotels" not in globals():
    hotels = pd.read_csv(HOTELS_URL)

def prepare_hotels(hotels):
    # Part B, Tasks 1-4, in one function
    hotels_nl = hotels.drop(columns=["reservation_status", "reservation_status_date"])
    hotels_mv = hotels_nl.copy()
    hotels_mv["children"] = hotels_mv["children"].fillna(0)
    hotels_mv["country"] = hotels_mv["country"].fillna("Unknown")
    hotels_mv["has_agent"] = hotels_mv["agent"].notna().astype(int)        # structural: booked without an agent
    hotels_mv["has_company"] = hotels_mv["company"].notna().astype(int)    # structural: booked without a company
    hotels_mv = hotels_mv.drop(columns=["agent", "company"])
    guests = hotels_mv["adults"] + hotels_mv["children"] + hotels_mv["babies"]
    hotels_q = hotels_mv[(guests > 0) & hotels_mv["adr"].between(0, 1000)].copy()
    hotels_fe = hotels_q.copy()
    hotels_fe["total_nights"] = hotels_fe["stays_in_weekend_nights"] + hotels_fe["stays_in_week_nights"]
    hotels_fe["total_guests"] = hotels_fe["adults"] + hotels_fe["children"] + hotels_fe["babies"]
    hotels_fe["is_family"] = ((hotels_fe["children"] + hotels_fe["babies"]) > 0).astype(int)
    hotels_fe["arrival_month"] = pd.to_datetime(hotels_fe["arrival_date_month"], format="%B").dt.month
    return hotels_fe

NUM_H = ["lead_time", "total_nights", "total_guests", "adr", "previous_cancellations",
         "previous_bookings_not_canceled", "booking_changes", "days_in_waiting_list",
         "required_car_parking_spaces", "total_of_special_requests", "arrival_month",
         "is_repeated_guest", "is_family", "has_agent", "has_company"]
CAT_H = ["hotel", "meal", "country", "market_segment", "distribution_channel",
         "deposit_type", "customer_type", "reserved_room_type"]
hotels_ready = prepare_hotels(hotels)
hotels_c = hotels_ready.sample(30_000, random_state=42)     # the same rows as `sample` in Part B

# ---- C2: pure noise
rng_noise = np.random.default_rng(42)
X_noise = rng_noise.normal(size=(100, 5000))
y_noise = rng_noise.permutation(np.repeat([0, 1], 50))
cv_noise = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ---- C3: bookings with a high-cardinality ID that carries no signal
c3 = hotels_c.copy()
rng_ids = np.random.default_rng(42)
c3["guest_id"] = pd.Series(rng_ids.integers(0, 10_000, size=len(c3)), index=c3.index).map(lambda i: f"G{i:05d}")
c3["agent"] = hotels.loc[c3.index, "agent"].fillna(0).astype(int).astype(str)        # real IDs; "0" = none
c3["company"] = hotels.loc[c3.index, "company"].fillna(0).astype(int).astype(str)
c3_train, c3_test = train_test_split(c3, test_size=0.2, stratify=c3["is_canceled"], random_state=42)
LOW_CAT = [c for c in CAT_H if c != "country"]

# ---- C6: five bookings the model has never seen, one with an unseen country and one with a missing price
MODEL_FILE = "hotel_model.joblib"
new_bookings = hotels_ready.drop(index=hotels_c.index).sample(5, random_state=7)[NUM_H + CAT_H].copy()
new_bookings.loc[new_bookings.index[1], "country"] = "XXX"        # a country code never seen in training
new_bookings.loc[new_bookings.index[3], "adr"] = np.nan           # a missing price

print(f"Part C ready ✔  Ames: {len(X_train):,} training houses, Part A CV RMSE = {cv_rmse_partA:.4f} · "
      f"hotels_c: {len(hotels_c):,} bookings · noise: {X_noise.shape[0]} × {X_noise.shape[1]:,} · "
      f"C3: {len(c3_train):,} train / {len(c3_test):,} test · C6: {len(new_bookings)} new bookings")

## C1 · Write your own transformer 🌶️🌶️ (15 min) · 💻

**Why it matters.** Every scikit-learn step follows one contract: `fit` learns, `transform` applies, `get_params` lists the settings. Write one yourself and scikit-learn becomes a toolbox you can extend with your own rules — re-fitted on every cross-validation fold, so they cannot leak.

**Build** `RareCategoryGrouper(min_count=50, other="Other")`, a subclass of `BaseEstimator` and `TransformerMixin`:
- `fit(X, y=None)` stores the column names in `self.feature_names_in_` and, per column, the categories seen at least `min_count` times in `self.frequent_` (a dict: column → set);
- `transform(X)` returns a DataFrame in which every other value — including categories never seen in training — becomes `other`;
- `get_feature_names_out(input_features=None)` returns the column names.

Then put it in front of `OneHotEncoder(handle_unknown="ignore")` for `CAT_H` in the Part B pipeline (`NUM_H` scaled, `LogisticRegression(max_iter=1000)`), and store the 5-fold CV ROC-AUC on `hotels_c` in `auc_grouped`. Store the same score with `OneHotEncoder(handle_unknown="ignore", min_frequency=50)` instead in `auc_minfreq`.

**Hints.** `__init__` may only store its arguments under the same names: `cross_val_score` *clones* the pipeline for every fold, and `clone()` rebuilds your object from `get_params()`. Learned attributes end with `_`. `s.where(s.isin(keep), "Other")` replaces everything outside `keep`.

In [ ]:
# 🚀 Challenge: Write RareCategoryGrouper (fit / transform / get_feature_names_out), use it in the Part B pipeline
#    on hotels_c, and store auc_grouped and auc_minfreq (see the instructions above).
# class RareCategoryGrouper(BaseEstimator, TransformerMixin):
#     def __init__(self, min_count=50, other="Other"):
#         self.min_count = min_count
#         self.other = other
#
#     def fit(self, X, y=None):
#         X = pd.DataFrame(X)
#         self.feature_names_in_ = ...
#         self.frequent_ = {...}           # column -> set of categories seen >= min_count times
#         return self
#
#     def transform(self, X):
#         ...                              # return a DataFrame
#
#     def get_feature_names_out(self, input_features=None):
#         return ...
#
# auc_grouped = cross_val_score(..., cv=5, scoring="roc_auc").mean()
# auc_minfreq = ...


In [ ]:
#@title ✅ Check — 🚀 C1 { display-mode: "form" }
def _check_c1():
    toy = pd.DataFrame({"c": ["a", "a", "a", "b", "b", "c"]})
    grouper = clone(RareCategoryGrouper(min_count=2))                  # clone() must work
    out = grouper.fit(toy).transform(pd.DataFrame({"c": ["a", "b", "c", "z"]}))
    return (list(out["c"]) == ["a", "b", "Other", "Other"]             # rare and unseen -> "Other"
            and list(grouper.get_feature_names_out()) == ["c"]
            and grouper.get_params() == {"min_count": 2, "other": "Other"}
            and abs(auc_grouped - auc_minfreq) < 0.005)

check("🚀 C1", _check_c1,
      hint="transform must return a DataFrame; fit stores feature_names_in_ and frequent_; "
           "__init__ only stores min_count and other. Compare both AUCs with the same cv.")

**✍️ Your answer** — Grouping did not raise the AUC. Why is `OneHotEncoder(min_frequency=...)` still worth having — and when would you write your own transformer?

> *(double-click this cell and write your answer here)*

## C2 · Selection outside cross-validation invents signal 🌶️🌶️ (10 min) · 📈💻

**Why it matters.** Choosing features by their relationship with the target is *learning from the labels*. Done once on all the data before cross-validation, the selection has already seen every validation fold — and with many features and few rows (genomics, text, sensor data) the illusion can be complete.

The setup cell created `X_noise`, **100 rows × 5,000 columns of pure random noise**, and `y_noise`, 50 zeros and 50 ones in random order. No column has anything to do with the target, so an honest model scores an AUC of about 0.5.

**Build** (a) the leaky way: `SelectKBest(f_classif, k=20)` fitted on **all** 100 rows, then `cross_val_score` of `LogisticRegression(max_iter=1000)` on the 20 selected columns → `auc_leaky`; (b) the honest way: the same two steps in one `Pipeline`, cross-validated on all 5,000 columns → `auc_honest`. Use `cv=cv_noise` and `scoring="roc_auc"` for both. Bonus: repeat with five fresh noise datasets — is it a fluke?

In [ ]:
# 🚀 Challenge: On X_noise / y_noise, compute auc_leaky (SelectKBest on all rows, then CV) and auc_honest
#    (SelectKBest inside a Pipeline). Use cv=cv_noise and scoring="roc_auc".
# X_top20 = SelectKBest(f_classif, k=20).fit_transform(X_noise, y_noise)
# auc_leaky = cross_val_score(LogisticRegression(max_iter=1000), X_top20, y_noise, cv=cv_noise, scoring="roc_auc").mean()
# honest = Pipeline([("select", ...), ("clf", ...)])
# auc_honest = ...


In [ ]:
#@title ✅ Check — 🚀 C2 { display-mode: "form" }
check('🚀 C2', lambda: auc_leaky > 0.85 and auc_honest < 0.7 and auc_leaky - auc_honest > 0.25, hint='Leaky: fit_transform the selector on X_noise first, then cross_val_score. Honest: cross_val_score the Pipeline on X_noise itself.')

**✍️ Your answer** — Why does pure noise score so well when the selection happens outside cross-validation? Where else can the same trap hide?

> *(double-click this cell and write your answer here)*

## C3 · Target encoding, the right way 🌶️🌶️ (15 min) · 💻

**Why it matters.** *Target (mean) encoding* replaces each category by the average target of its rows: one number instead of hundreds of one-hot columns, which is ideal for high-cardinality columns. But a category with only a few rows is encoded mostly by its **own** labels — the model is shown the answer. Customer IDs, ZIP codes and product codes behave exactly like this.

The setup cell split `hotels_c` into `c3_train` / `c3_test` (80/20) and added `guest_id`: 10,000 random IDs (about 2.6 bookings per ID in training) that carry **no** information about cancellations — a stand-in for a customer ID.

**Build** two logistic regressions (`max_iter=1000`) on `NUM_H` (scaled) and `LOW_CAT` (one-hot, `min_frequency=50`) plus `country` and `guest_id`, encoded
- (a) **naively**: per category, the mean of `is_canceled` over **all** of `c3_train`, mapped onto train and test (unseen categories → the overall rate) → `auc_naive_train`, `auc_naive_test`;
- (b) with `TargetEncoder()` inside the `ColumnTransformer` → `auc_te_train`, `auc_te_test` (ROC-AUC on `c3_train` and `c3_test`).

**Hints.** `y.groupby(X[col]).mean()` gives the category means and `.map(means)` applies them. `TargetEncoder` *cross-fits* in `fit_transform`: each fold of the training data is encoded with means from the other folds, so no row sees its own label — and a `Pipeline` calls `fit_transform` for you. For reproducible numbers call `np.random.seed(42)` right before fitting (TargetEncoder shuffles its folds with NumPy's global random generator). Bonus: repeat with the real IDs `country`, `agent` and `company`.

In [ ]:
# 🚀 Challenge: Compare naive mean encoding with TargetEncoder for country + guest_id on c3_train / c3_test and store
#    auc_naive_train, auc_naive_test, auc_te_train and auc_te_test.
# y_tr, y_te = c3_train["is_canceled"], c3_test["is_canceled"]
# means = y_tr.groupby(c3_train["guest_id"]).mean()
# ...
# auc_naive_train = roc_auc_score(y_tr, naive.predict_proba(...)[:, 1])
# ...


In [ ]:
#@title ✅ Check — 🚀 C3 { display-mode: "form" }
check('🚀 C3', lambda: auc_naive_train - auc_naive_test > 0.05 and auc_te_test - auc_naive_test > 0.03 and abs(auc_te_train - auc_te_test) < 0.02, hint='Naive: learn the means on c3_train, map them onto both tables (fillna for unseen IDs). TargetEncoder: put it in the ColumnTransformer and fit the pipeline on c3_train.')

**✍️ Your answer** — Why does naive encoding make the model *worse* on new bookings — and why do `country`, `agent` and `company` hardly suffer from it?

> *(double-click this cell and write your answer here)*

## C4 · Imputation shoot-out 🌶️🌶️ (15 min) · 📈

**Why it matters.** Tutorials love fancy imputers. Does the choice matter for *your* model? Don't guess — measure, and compare the differences with the fold-to-fold noise.

**Build** a dict `imputation_rmse` (name → mean 5-fold CV RMSE on `X_train`, `y_log`, `cv=5` as in Part A) for five ways to handle the missing numeric values: `"median"`, `"median + indicator"`, `"KNN"` (`KNNImputer(n_neighbors=5)`) and `"iterative"` (`IterativeImputer(random_state=42, skip_complete=True)`), each followed by `Ridge(alpha=10)` — and `"HGB"`: **no imputation at all**, but a `HistGradientBoostingRegressor(random_state=42)`, which routes missing values itself.

**Hints.** `make_preprocess(numeric)` from the setup cell builds the Part A preprocessing with your own numeric pipeline, e.g. `Pipeline([("scale", StandardScaler()), ("impute", KNNImputer())])` — scale *before* KNN, so distances are fair. For HGB, pass the numeric columns through unchanged (`"passthrough"`) and turn the text columns into integer codes (`OrdinalEncoder`). Keep the per-fold scores to compare methods fold by fold. Bonus: hide 30 % of `OverallQual` at random — does imputation matter now?

In [ ]:
# 🚀 Challenge: Build imputation_rmse: CV RMSE for "median", "median + indicator", "KNN", "iterative" (each + Ridge)
#    and "HGB" (HistGradientBoostingRegressor, no imputation).
# def ridge_with(imputer):
#     numeric = Pipeline([("scale", StandardScaler()), ("impute", imputer)])
#     return Pipeline([("prep", make_preprocess(numeric)), ("reg", Ridge(alpha=10))])
#
# hgb = Pipeline([("prep", ColumnTransformer([("num", "passthrough", NUMERIC_COLS), ...])), ...])
# imputation_rmse = {...}


In [ ]:
#@title ✅ Check — 🚀 C4 { display-mode: "form" }
check('🚀 C4', lambda: len(pd.Series(imputation_rmse)) == 5 and pd.Series(imputation_rmse).between(0.10, 0.15).all() and abs(pd.Series(imputation_rmse).min() - 0.1144) < 0.003, hint="Five entries, each the mean of -cross_val_score(..., cv=5, scoring='neg_root_mean_squared_error').")

**✍️ Your answer** — Which way of handling the missing values wins — and is the difference real?

> *(double-click this cell and write your answer here)*

## C5 · Train in logs, report in dollars 🌶️ (10 min) · 📈

**Why it matters.** We trained on log prices because errors then count *relatively*. Managers, however, think in dollars. `TransformedTargetRegressor` trains on `log1p(y)` and back-transforms every prediction with `expm1`, so `predict` and `cross_val_score` work in dollars directly.

**Build** `raw_model`, the Part A pipeline (`make_preprocess()` + `Ridge(alpha=10)`) trained on `SalePrice` in dollars, and `log_model = TransformedTargetRegressor(regressor=<the same pipeline>, func=np.log1p, inverse_func=np.expm1)`. Store their 5-fold CV mean absolute error in dollars (`scoring="neg_mean_absolute_error"`, target `train_fe["SalePrice"]`) in `mae_raw` and `mae_log`. Bonus: compare the errors for cheap and expensive houses (`cross_val_predict`).

In [ ]:
# 🚀 Challenge: Store the CV MAE in dollars of a Ridge pipeline trained on raw prices (mae_raw) and of the same
#    pipeline wrapped in TransformedTargetRegressor with log1p / expm1 (mae_log).
# y_price = train_fe["SalePrice"]
# raw_model = Pipeline([("prep", make_preprocess()), ("reg", Ridge(alpha=10))])
# log_model = TransformedTargetRegressor(regressor=..., func=np.log1p, inverse_func=np.expm1)
# mae_raw = -cross_val_score(raw_model, X_train, y_price, cv=5, scoring="neg_mean_absolute_error").mean()
# mae_log = ...


In [ ]:
#@title ✅ Check — 🚀 C5 { display-mode: "form" }
check('🚀 C5', lambda: 10_000 < mae_log < 20_000 and mae_raw - mae_log > 1_000, hint='Both models get the target in dollars; only log_model transforms it internally.')

**✍️ Your answer** — Why does the log model win even when the errors are measured in dollars?

> *(double-click this cell and write your answer here)*

## C6 · Ship the pipeline: save, load and validate 🌶️🌶️ (15 min) · 💻

**Why it matters.** A model is only useful once another program can call it, for example a batch job that scores tonight's new bookings. A fitted pipeline carries every learned step with it, and a thin validation layer in front of it turns broken input into a clear error instead of a cryptic message deep inside scikit-learn, or worse, a silent number.

The setup cell prepared `new_bookings`: five bookings the model has never seen (the `NUM_H` + `CAT_H` columns), one with a country code that never occurs in training (`"XXX"`) and one with a missing price (`adr` is NaN). The Part B pipeline would crash on that missing price.

**Build**
1. `ship_pipe`: the Part B pipeline with a `SimpleImputer(strategy="median")` in front of the scaler, fitted on all of `hotels_c`. Save it with `joblib.dump(ship_pipe, MODEL_FILE)` and load it back into `loaded_pipe`.
2. `score_bookings(new_df)`: raise a `ValueError` that names the problem columns when columns of `NUM_H + CAT_H` are missing or a `NUM_H` column is not numeric; otherwise return a Series of cancellation probabilities with `new_df`'s index.
3. Score `new_bookings`, and confirm that `loaded_pipe` predicts exactly what `ship_pipe` predicts.

**Hints.** `pd.api.types.is_numeric_dtype(s)` tests a column's type. `OneHotEncoder(handle_unknown="ignore")` already copes with unseen categories. A joblib file is a pickle: load only files you trust, because unpickling can run code. Bonus: also reject impossible values, e.g. Part B's rule that `adr` lies between 0 and 1000.

In [ ]:
# 🚀 Challenge: Fit and save ship_pipe, reload it as loaded_pipe, and write score_bookings(new_df) with input validation
#    (see the instructions above).
# import joblib
# FEATURES = NUM_H + CAT_H
# ship_pipe = Pipeline([("prep", ColumnTransformer([
#     ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), NUM_H),
#     ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=50), CAT_H)])),
#     ("clf", LogisticRegression(max_iter=1000))]).fit(...)
# joblib.dump(ship_pipe, MODEL_FILE)
# loaded_pipe = joblib.load(MODEL_FILE)
#
# def score_bookings(new_df, model=loaded_pipe):
#     missing_cols = [c for c in FEATURES if c not in new_df.columns]
#     if missing_cols:
#         raise ValueError(f"missing columns: {missing_cols}")
#     ...                                   # numeric check for NUM_H, then predict_proba
#     return pd.Series(..., index=new_df.index, name="p_cancel")


In [ ]:
#@title ✅ Check — 🚀 C6 { display-mode: "form" }
def _check_c6():
    import joblib
    p = score_bookings(new_bookings)                                    # ⏳ until score_bookings exists
    X_new = new_bookings[NUM_H + CAT_H]
    same = np.array_equal(joblib.load(MODEL_FILE).predict_proba(X_new), ship_pipe.predict_proba(X_new))
    scores_ok = (isinstance(p, pd.Series) and p.index.equals(new_bookings.index)
                 and p.notna().all() and p.between(0, 1).all())
    clear = []
    for broken, col in [(new_bookings.drop(columns="lead_time"), "lead_time"),
                        (new_bookings.assign(adr="95 EUR"), "adr")]:
        try:
            score_bookings(broken)
            clear.append(False)                                         # broken input must not be scored
        except ValueError as err:
            clear.append(col in str(err))                               # ... and the message names the column
    return same and scores_ok and all(clear)

check("🚀 C6", _check_c6,
      hint="Reload MODEL_FILE with joblib.load; score_bookings must return a Series of probabilities for all five "
           "bookings and raise ValueError (naming the column) for a missing column or text in a numeric column.")

**✍️ Your answer** — Your batch job will score real bookings every night. What else does it need before you can trust it?

> *(double-click this cell and write your answer here)*

## 🏁 Open challenge · Beat the Part A pipeline — without leakage (20 min) · 📈💻

The Part A pipeline scores a CV RMSE of **0.1144** (`cv_rmse_partA`). Beat it. **Rules:** use only `X_train` / `y_log` and the same `cross_val_score(..., cv=5, scoring="neg_root_mean_squared_error")`; every learned step lives inside the pipeline; **no look at `test`** while you experiment. Store your best pipeline in `best_pipe` and its CV RMSE in `best_cv_rmse`, then report the **one** idea that helped most.

Ideas: log-transform skewed size features · treat code-like numbers as categories (`MSSubClass` is a dwelling type, not a quantity!) · tune `alpha` (`RidgeCV`) · interactions · a different model.

In [ ]:
# 🚀 🏁 Open challenge: Improve the Part A pipeline's CV RMSE without leakage; store best_pipe and best_cv_rmse.
# num_step, ord_step, nom_step = (step for _, step, _ in make_preprocess().transformers)
# best_pipe = Pipeline([("prep", ...), ("reg", ...)])
# best_cv_rmse = -cross_val_score(best_pipe, X_train, y_log, cv=5, scoring="neg_root_mean_squared_error").mean()


In [ ]:
#@title ✅ Check — 🚀 🏁 Open challenge { display-mode: "form" }
check('🚀 🏁 Open challenge', lambda: best_cv_rmse < cv_rmse_partA, hint="Same cross_val_score(..., X_train, y_log, cv=5, scoring='neg_root_mean_squared_error') — and mind the minus sign.")

**✍️ Your answer** — Report your best CV RMSE and the **one** idea that helped most.

> *(double-click this cell and write your answer here)*

In [ ]:
progress()

---
## Going further (optional, self-study)

- `make_column_selector(dtype_include="number")` picks columns by type, so you need no hand-written column lists.
- `sklearn.set_config(transform_output="pandas")` makes every transformer return DataFrames with readable names (dense outputs only — see side quest A6).
- `PowerTransformer(method="yeo-johnson")` for skewed features, including zeros and negative values.
- `SequentialFeatureSelector` (wrapper selection) — slower than filters, but it takes interactions into account.
- Already in the 🚀 lane: `IsolationForest` (side quest A3), `FunctionTransformer` (A4), your own transformer (C1), `TargetEncoder` (C3), `IterativeImputer` (C4), `TransformedTargetRegressor` (C5), saving a model with `joblib` (C6).
- Read: Kuhn & Johnson, *Feature Engineering and Selection* (free online at feat.engineering).

**Data sources and licences:** Ames Housing — De Cock (2011), *Journal of Statistics Education* 19(3), via OpenML (data id 42165), educational use · Hotel Booking Demand — Antonio, de Almeida & Nunes (2019), *Data in Brief* 22, CC BY 4.0. The noise data (C2) and `guest_id` (C3) are generated in the notebook.